In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent

## Tabular

In [2]:
df = pd.read_csv(f'{ROOT}/data/isic-2024-challenge/train-metadata.csv')
df = df.ffill()
display(df.head(2))
print(df.shape)

/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_19699/1929093370.py:1: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{ROOT}/data/isic-2024-challenge/train-metadata.csv')


,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455


(401059, 55)


split the dataset into train validation test

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True)

TEST_SPLIT = 0.1
N_SPLITS = int(1 / TEST_SPLIT)
RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(X=np.zeros(len(df)), y=df['target'], groups=df['patient_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [7]:
assert set(train_df["patient_id"]).isdisjoint(set(test_df["patient_id"]))  # no group overlap
print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Class dist (train):")
print(train_df["target"].value_counts(normalize=True))
print("Class dist (test):")
print(test_df["target"].value_counts(normalize=True))

Train size: 363483 | Test size: 37576
Class dist (train):
target
0    0.998999
1    0.001001
Name: proportion, dtype: float64
Class dist (test):
target
0    0.999228
1    0.000772
Name: proportion, dtype: float64


In [ ]:
path = f"{ROOT}/data/new/"

train_df.to_csv(path + "train-metadata.csv", index=False)
test_df.to_csv(path + "test-metadata.csv", index=False)

## Images

In [13]:
path = f"{ROOT}/data/new/"

train_df = pd.read_csv(path + "train-metadata.csv")
test_df = pd.read_csv(path + "test-metadata.csv")

/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_1061/2727820117.py:3: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(path + "train-metadata.csv")


In [ ]:
from PIL import Image

img = Image.open('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_0015670.jpg')
img.show()
print(img.size)

(139, 139)


In [ ]:
patient_ids_test = set(test_df['isic_id'].to_list())
patient_ids_train = set(train_df['isic_id'].to_list())

In [ ]:
import shutil
from pathlib import Path

def create_new_img_dataset(img_folder, patient_ids, out_folder):
    img_folder = Path(img_folder)
    out_folder = Path(out_folder)
    out_folder.mkdir(parents=True, exist_ok=True)
    
    new_map = []
    for file in img_folder.glob('*.jpg'):
        if file.stem in patient_ids:
            shutil.copy(file, out_folder / file.name)
            new_map.append(file.name)
            
    return new_map

IMG_FOLDER = f"{ROOT}/data/isic-2024-challenge/train-image/image"
OUT_FOLDER_TRAIN = f"{ROOT}/data/new/train-image"
OUT_FOLDER_TEST = f"{ROOT}/data/new/test-image"

# train_images = create_new_img_dataset(IMG_FOLDER, patient_ids_train, OUT_FOLDER_TRAIN)
test_images = create_new_img_dataset(IMG_FOLDER, patient_ids_test, OUT_FOLDER_TEST)

In [ ]:
len(test_images)


37576

In [ ]:
import 

def create_new_bytestream_dataset(in_filepath, patient_ids, out_filepath):
    in_filepath = Path(in_filepath)
    out_filepath = Path(out_filepath)
    out_filepath.parent.mkdir(parents=True, exist_ok=True)
    copied, missing = 0, 0
    
    with h5py.File(in_filepath, "r") as src, h5py.File(out_filepath, "w") as dst:
        # copy file-level attributes
        for k, v in src.attrs.items():
            dst.attrs[k] = v

        for pid in patient_ids:
            if pid in src:
                src.copy(src[pid], dst)
                copied += 1
            else:
                missing += 1

    print(f"Copied: {copied}, Missing: {missing}")
    return

in_filepath = f"{ROOT}/data/isic-2024-challenge/train-image.hdf5"
out_filepath_train = f"{ROOT}/data/new/train-image.hdf5"
out_filepath_test = f"{ROOT}/data/new/test-image.hdf5"

create_new_bytestream_dataset(in_filepath,patient_ids_test,out_filepath_test)
create_new_bytestream_dataset(in_filepath,patient_ids_train,out_filepath_train)

Copied: 37576, Missing: 0
Copied: 363483, Missing: 0


## Tabular Dataset Enhancement

In [20]:
def feature_engineering(df):
    # New features to try...
    df["lesion_size_ratio"] = df["tbp_lv_minorAxisMM"] / df["clin_size_long_diam_mm"]
    df["lesion_shape_index"] = df["tbp_lv_areaMM2"] / (df["tbp_lv_perimeterMM"] ** 2)
    df["hue_contrast"] = (df["tbp_lv_H"] - df["tbp_lv_Hext"]).abs()
    df["luminance_contrast"] = (df["tbp_lv_L"] - df["tbp_lv_Lext"]).abs()
    df["lesion_color_difference"] = np.sqrt(df["tbp_lv_deltaA"] ** 2 + df["tbp_lv_deltaB"] ** 2 + df["tbp_lv_deltaL"] ** 2)
    df["border_complexity"] = df["tbp_lv_norm_border"] + df["tbp_lv_symm_2axis"]
    df["color_uniformity"] = df["tbp_lv_color_std_mean"] / df["tbp_lv_radial_color_std_max"]
    df["3d_position_distance"] = np.sqrt(df["tbp_lv_x"] ** 2 + df["tbp_lv_y"] ** 2 + df["tbp_lv_z"] ** 2) 
    df["perimeter_to_area_ratio"] = df["tbp_lv_perimeterMM"] / df["tbp_lv_areaMM2"]
    df["lesion_visibility_score"] = df["tbp_lv_deltaLBnorm"] + df["tbp_lv_norm_color"]
    df["combined_anatomical_site"] = df["anatom_site_general"] + "_" + df["tbp_lv_location"]
    df["symmetry_border_consistency"] = df["tbp_lv_symm_2axis"] * df["tbp_lv_norm_border"]
    df["color_consistency"] = df["tbp_lv_stdL"] / df["tbp_lv_Lext"]
    
    df["size_age_interaction"] = df["clin_size_long_diam_mm"] * df["age_approx"]
    df["hue_color_std_interaction"] = df["tbp_lv_H"] * df["tbp_lv_color_std_mean"]
    df["lesion_severity_index"] = (df["tbp_lv_norm_border"] + df["tbp_lv_norm_color"] + df["tbp_lv_eccentricity"]) / 3
    df["shape_complexity_index"] = df["border_complexity"] + df["lesion_shape_index"]
    df["color_contrast_index"] = df["tbp_lv_deltaA"] + df["tbp_lv_deltaB"] + df["tbp_lv_deltaL"] + df["tbp_lv_deltaLBnorm"]
    df["log_lesion_area"] = np.log(df["tbp_lv_areaMM2"] + 1)
    df["normalized_lesion_size"] = df["clin_size_long_diam_mm"] / df["age_approx"]
    df["mean_hue_difference"] = (df["tbp_lv_H"] + df["tbp_lv_Hext"]) / 2
    df["std_dev_contrast"] = np.sqrt((df["tbp_lv_deltaA"] ** 2 + df["tbp_lv_deltaB"] ** 2 + df["tbp_lv_deltaL"] ** 2) / 3)
    df["color_shape_composite_index"] = (df["tbp_lv_color_std_mean"] + df["tbp_lv_area_perim_ratio"] + df["tbp_lv_symm_2axis"]) / 3
    df["3d_lesion_orientation"] = np.arctan2(df["tbp_lv_y"], df["tbp_lv_x"])
    df["overall_color_difference"] = (df["tbp_lv_deltaA"] + df["tbp_lv_deltaB"] + df["tbp_lv_deltaL"]) / 3
    df["symmetry_perimeter_interaction"] = df["tbp_lv_symm_2axis"] * df["tbp_lv_perimeterMM"]
    df["comprehensive_lesion_index"] = (df["tbp_lv_area_perim_ratio"] + df["tbp_lv_eccentricity"] + df["tbp_lv_norm_color"] + df["tbp_lv_symm_2axis"]) / 4

    new_num_cols = [
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "color_uniformity", "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index", 
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index",
    ]
    new_cat_cols = ["combined_anatomical_site"]

    return df, new_num_cols, new_cat_cols

train_df_enh, new_num_cols, new_cat_cols = feature_engineering(train_df.copy())
test_df_enh, _, _ = feature_engineering(test_df.copy())

In [8]:
import os

path = f"{ROOT}/data/new"
os.makedirs(path, exist_ok=True)

In [ ]:
train_df.to_csv(os.path.join(path, "enhanced_train_metadata.gz"), index=False, compression='gzip')
test_df.to_csv(os.path.join(path, "enhanced_test_metadata.gz"), index=False, compression='gzip')

## Clean tabular dataset

In [17]:
train_df = pd.read_csv(os.path.join(path, "enhanced_train_metadata.csv")).ffill()
test_df = pd.read_csv(os.path.join(path, "enhanced_test_metadata.csv")).ffill()

/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_19699/3238614661.py:1: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(os.path.join(path, "enhanced_train_metadata.csv")).ffill()
/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_19699/3238614661.py:2: DtypeWarning: Columns (51) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(os.path.join(path, "enhanced_test_metadata.csv")).ffill()


In [18]:
def clean_df(df):
    EPS = 1e-6
    cleaned_df = df.copy()

    # --- sanitize color_uniformity ---
    cleaned_df["color_uniformity"] = (
        cleaned_df["tbp_lv_color_std_mean"] / (cleaned_df["tbp_lv_radial_color_std_max"] + EPS))

    cleaned_df["color_uniformity"] = (
        cleaned_df["color_uniformity"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype("float32"))

    return cleaned_df

In [23]:
train_df_enh_clean = clean_df(train_df)
test_df_enh_clean = clean_df(test_df)

train_df_enh_clean.drop(columns=["mel_thick_mm", "mel_mitotic_index"], inplace=True)
test_df_enh_clean.drop(columns=["mel_thick_mm", "mel_mitotic_index"], inplace=True)

# train_df_enh_clean.to_csv(os.path.join(path, "enhanced_train_metadata_clean.gz"), index=False, compression='gzip')
# test_df_enh_clean.to_csv(os.path.join(path, "enhanced_test_metadata_clean.gz"), index=False, compression='gzip')

train_df_enh_clean.to_parquet(os.path.join(path, "enhanced_train_metadata_clean.parquet"), engine='fastparquet', compression="zstd")
test_df_enh_clean.to_parquet(os.path.join(path, "enhanced_test_metadata_clean.parquet"), engine='fastparquet', compression="zstd")
